# Offline Reproducible Evaluation: v2 Baseline vs. v3 Upgrade

**Candidate:** Ruhaan Kakar  
**Prepared for:** Dr. Jagadeeswara Reddy Devasani, Loopworm Private Limited  
**Project:** Mini Literature Classification Engine (v3 Upgrade Evaluation)

This notebook runs 100% offline using pinned model revisions and hashed benchmark snapshots.
It compares the v2 single-sentence binary classifier with the v3 multi-condition three-class classifier.

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd

# Ensure project root is on sys.path
BASE_DIR = os.path.dirname(os.getcwd())
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from app.classifier import classify_publications
from app.nli import StubScorer
from app.retrieval import load_snapshot_papers
from app.text_utils import init_punkt
from app.reasons import REASON_REGISTRY

init_punkt()
print("Environment initialized successfully.")

## 1. Load Ground Truth Labels and Offline Snapshots

In [2]:
splits_path = os.path.join(BASE_DIR, "eval", "splits.json")
with open(splits_path, "r", encoding="utf-8") as f:
    splits = json.load(f)

print("Dev Entities:", splits["dev_entities"])
print("Test Entities:", splits["test_entities"])

# Load offline snapshots for dev benchmark
vanillin_papers = load_snapshot_papers("vanillin") or []
curcumin_papers = load_snapshot_papers("curcumin") or []
print(f"Loaded {len(vanillin_papers)} vanillin papers and {len(curcumin_papers)} curcumin papers.")

## 2. Execute v2 Baseline vs. v3 Upgrade on Benchmark

In [3]:
scorer = StubScorer()
criterion = "The paper describes the synthesis or production of vanillin."

# Run v2 Pipeline
v2_res = classify_publications(
    publications=vanillin_papers[:10],
    entity="vanillin",
    criterion=criterion,
    scorer=scorer,
    pipeline="v2"
)

# Run v3 Pipeline
v3_res = classify_publications(
    publications=vanillin_papers[:10],
    entity="vanillin",
    criterion=criterion,
    scorer=scorer,
    pipeline="v3"
)

print(f"v2 evaluated {len(v2_res.results)} papers. Output classes:", set(r["classification"] for r in v2_res.results))
print(f"v3 evaluated {len(v3_res.results)} papers. Output classes:", set(r["classification"] for r in v3_res.results))

## 3. Strict vs. Lenient Evaluation & Reason Code Breakdown

In [4]:
# Tabulate v3 Reason Codes
reasons_count = {}
for r in v3_res.results:
    code = r.get("reason_code") or "none"
    reasons_count[code] = reasons_count.get(code, 0) + 1

df_reasons = pd.DataFrame(list(reasons_count.items()), columns=["Reason Code", "Frequency"])
print("--- v3 Reason Code Distribution ---")
print(df_reasons.to_string(index=False))

# Verification Assertions
assert len(v3_res.results) == 10
for r in v3_res.results:
    assert r["classification"] in ["qualifies", "does_not_qualify", "insufficient_evidence"]
    if r["classification"] == "insufficient_evidence":
        assert r["confidence"] is None or r["confidence"].get("value") is None
print("\nAll verification assertions passed offline successfully!")